## Single frame metrics

In [1]:
import open3d as o3d
import numpy as np

###  Point density & density variance (Nearest neighbor)

In [ ]:
def density_metrics(pcd, k=10):
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    pts = np.asarray(pcd.points)
    dists = []
    for p in pts:
        [_, idx, d] = pcd_tree.search_knn_vector_3d(p, k + 1)
        dists.append(np.mean(np.sqrt(d[1:])))  # exclude self
    dists = np.array(dists)
    
    print(f"mean_nn_dist_mm: { np.mean(dists)} \n density_variance: {np.var(dists)} \n density_std: {np.std(dists)}")
    return {
        "mean_nn_dist_mm": np.mean(dists),
        "density_variance": np.var(dists),
        "density_std": np.std(dists),
    }

### Normal consistency

In [ ]:
def normal_consistency(pcd, k=10):
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30)
    )
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    pts = np.asarray(pcd.points)
    normals = np.asarray(pcd.normals)
    scores = []
    for i, p in enumerate(pts):
        [_, idx, _] = pcd_tree.search_knn_vector_3d(p, k + 1)
        neighbours = idx[1:]
        cos_sims = np.abs(normals[neighbours] @ normals[i])
        scores.append(np.mean(cos_sims))
        
    print(f"\n Normal consistency {np.mean(scores)}")
    return np.mean(scores)  # 0–1, higher is smoother

### Local urvature

In [ ]:
def local_curvature(pcd, k=20):
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    pts = np.asarray(pcd.points)
    curvatures = []
    for p in pts:
        [_, idx, _] = pcd_tree.search_knn_vector_3d(p, k + 1)
        neighbours = pts[idx[1:]]
        cov = np.cov((neighbours - neighbours.mean(axis=0)).T)
        eigvals = np.linalg.eigvalsh(cov)
        eigvals = np.sort(np.abs(eigvals))
        curvature = eigvals[0] / (eigvals.sum() + 1e-8)
        curvatures.append(curvature)
        
    print(f"\nmean_curvature: {np.mean(curvatures)} \n max_curvature: {np.max(curvatures)} \n curvature_std: {np.std(curvatures)}")
    return {
        "mean_curvature": np.mean(curvatures),
        "max_curvature": np.max(curvatures),
        "curvature_std": np.std(curvatures),
    }

## Results

In [6]:
def evaluate(pcd):
    print("Results for PC: \n")
    density_metrics(pcd)
    normal_consistency(pcd)
    local_curvature(pcd)

pcd = o3d.io.read_point_cloud("data/cloud_10_frame50.ply")
evaluate(pcd)

Results for PC: 

